# Snorkel for Object Detection: Weak Supervision on Pascal VOC 2012

This tutorial demonstrates how to use **Snorkel** (a programmatic data labeling framework) to bootstrap object detection with weak supervision.

## Pipeline Overview

```
VOC 2012 images → Selective Search proposals → Snorkel LFs → LabelModel
→ pseudo-boxes → COCO format → fine-tune DETR → evaluate mAP
```

## Dataset: Pascal VOC 2012
- **4 selected classes:** aeroplane, boat, bottle, pottedplant
- **~11,530 images** with difficult but visually distinct objects
- Each class has distinct visual properties (color, shape, position) suited to heuristic labeling

In [ ]:
# @title Install dependencies
!pip install -q snorkel transformers torch torchvision opencv-contrib-python matplotlib seaborn scikit-learn scipy pycocotools
!pip install -q gitpython

In [ ]:
import os
import json
import xml.etree.ElementTree as ET
import random
from pathlib import Path

import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns
import pandas as pd
from tqdm import tqdm

from functools import partial

import torch
from torch.utils.data import Dataset, DataLoader

import transformers
from transformers import DetrForObjectDetection, DetrImageProcessor
from transformers import Trainer, TrainingArguments

import snorkel
from snorkel.labeling import labeling_function, LFAnalysis
from snorkel.labeling.model import LabelModel, MajorityLabelVoter
from snorkel.labeling import PandasLFApplier

from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, precision_recall_curve, average_precision_score

print("All imports successful!")
print(f"Snorkel version: {snorkel.__version__}")
print(f"Transformers version: {transformers.__version__}")
print(f"Torch version: {torch.__version__}")
print(f"OpenCV version: {cv2.__version__}")

# Set plot style
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 100

## 2. Dataset Download & Exploration

We download Pascal VOC 2012 and parse the XML annotations. We select 4 visually distinct classes (aeroplane, boat, bottle, pottedplant) that are well-suited to heuristic labeling.

In [ ]:
# @title Download Pascal VOC 2012
import torchvision
from torchvision.datasets import VOCDetection

DATA_DIR = "voc_dataset"
VOC_ROOT = os.path.join(DATA_DIR, "VOCdevkit")

if not os.path.exists(VOC_ROOT):
    print("Downloading Pascal VOC 2012 (~2GB)...")
    voc_dataset = VOCDetection(root=DATA_DIR, year="2012", image_set="trainval", download=True)
    print("Download complete!")
else:
    print("VOC dataset already exists!")

# Set image and annotation directories
IMAGES_DIR = os.path.join(VOC_ROOT, "VOC2012", "JPEGImages")
ANNOTATIONS_DIR = os.path.join(VOC_ROOT, "VOC2012", "Annotations")

# Check structure
!ls {ANNOTATIONS_DIR} | head -5

In [ ]:
# Parse VOC annotations (VOC format XML)
CLASS_MAP = {"aeroplane": 0, "boat": 1, "bottle": 2, "pottedplant": 3}
CLASS_NAMES = {0: "aeroplane", 1: "boat", 2: "bottle", 3: "pottedplant"}

def parse_voc_xml(xml_path):
    tree = ET.parse(xml_path)
    root = tree.getroot()
    filename = root.find("filename").text
    if not filename.lower().endswith((".jpg", ".jpeg", ".png", ".bmp")):
        filename += ".jpg"
    size = root.find("size")
    width = int(size.find("width").text)
    height = int(size.find("height").text)
    boxes = []
    for obj in root.findall("object"):
        name = obj.find("name").text
        if name not in CLASS_MAP:
            continue
        bbox = obj.find("bndbox")
        xmin = int(float(bbox.find("xmin").text))
        ymin = int(float(bbox.find("ymin").text))
        xmax = int(float(bbox.find("xmax").text))
        ymax = int(float(bbox.find("ymax").text))
        boxes.append({
            "class_id": CLASS_MAP[name],
            "class_name": name,
            "bbox": [xmin, ymin, xmax, ymax],
        })
    return {"filename": filename, "width": width, "height": height, "boxes": boxes}

# Parse all annotations
all_annotations = {}
xml_files = sorted(Path(ANNOTATIONS_DIR).glob("*.xml"))
for xml_path in xml_files:
    ann = parse_voc_xml(xml_path)
    all_annotations[ann["filename"]] = ann

print(f"Total images with selected class annotations: {len(all_annotations)}")

# Class distribution
class_counts = {}
for cls_name in CLASS_MAP:
    class_counts[cls_name] = 0
for ann in all_annotations.values():
    for box in ann["boxes"]:
        class_counts[box["class_name"]] += 1

print(f"\nClass distribution:")
for cls, count in class_counts.items():
    print(f"  {cls}: {count}")

# Image statistics
widths = [ann["width"] for ann in all_annotations.values()]
heights = [ann["height"] for ann in all_annotations.values()]
print(f"\nImage sizes: {min(widths)}x{min(heights)} to {max(widths)}x{max(heights)}")

In [ ]:
# Visualize sample images with ground truth boxes
def visualize_image(filename, ax):
    img_path = os.path.join(IMAGES_DIR, filename)
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    ax.imshow(img)
    ann = all_annotations[filename]
    colors = {"aeroplane": "cyan", "boat": "yellow", "bottle": "magenta", "pottedplant": "lime"}
    for box in ann["boxes"]:
        xmin, ymin, xmax, ymax = box["bbox"]
        rect = patches.Rectangle(
            (xmin, ymin), xmax - xmin, ymax - ymin,
            linewidth=2, edgecolor=colors[box["class_name"]],
            facecolor="none"
        )
        ax.add_patch(rect)
        ax.text(xmin, ymin - 5, box["class_name"],
                color=colors[box["class_name"]], fontsize=10, weight="bold")

sample_files = random.sample(list(all_annotations.keys()), 6)
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
for ax, fname in zip(axes.flat, sample_files):
    visualize_image(fname, ax)
    ax.axis("off")
plt.suptitle("Sample images with GT boxes (Cyan=Aeroplane, Yellow=Boat, Magenta=Bottle, Lime=Pottedplant)", fontsize=14)
plt.tight_layout()
plt.show()

## 3. Region Proposals via Selective Search

We use OpenCV's Selective Search to generate candidate bounding boxes. These proposals serve as the input for our Snorkel labeling functions.

In [ ]:
# @title Selective Search Proposal Generation
def generate_proposals(img, max_proposals=500):
    ss = cv2.ximgproc.segmentation.createSelectiveSearchSegmentation()
    ss.setBaseImage(img)
    ss.switchToSelectiveSearchFast()
    rects = ss.process()
    rects = rects[:max_proposals]
    proposals = []
    for x, y, w, h in rects:
        proposals.append([x, y, x + w, y + h])
    return proposals

def compute_iou(box_a, box_b):
    xA = max(box_a[0], box_b[0])
    yA = max(box_a[1], box_b[1])
    xB = min(box_a[2], box_b[2])
    yB = min(box_a[3], box_b[3])
    inter = max(0, xB - xA) * max(0, yB - yA)
    area_a = (box_a[2] - box_a[0]) * (box_a[3] - box_a[1])
    area_b = (box_b[2] - box_b[0]) * (box_b[3] - box_b[1])
    iou = inter / float(area_a + area_b - inter + 1e-10)
    return iou

In [ ]:
# Visualize proposals on a sample image
sample_fname = sample_files[0]
img_path = os.path.join(IMAGES_DIR, sample_fname)
img = cv2.imread(img_path)
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
proposals = generate_proposals(img, max_proposals=200)

fig, axes = plt.subplots(1, 2, figsize=(18, 8))
axes[0].imshow(img_rgb)
ann = all_annotations[sample_fname]
for box in ann["boxes"]:
    xmin, ymin, xmax, ymax = box["bbox"]
    rect = patches.Rectangle(
        (xmin, ymin), xmax - xmin, ymax - ymin,
        linewidth=2, edgecolor="lime", facecolor="none"
    )
    axes[0].add_patch(rect)
axes[0].set_title(f"GT Boxes ({len(ann['boxes'])} objects)", fontsize=12)
axes[0].axis("off")

# Plot some proposals
axes[1].imshow(img_rgb)
for i, prop in enumerate(proposals[:50]):
    xmin, ymin, xmax, ymax = prop
    rect = patches.Rectangle(
        (xmin, ymin), xmax - xmin, ymax - ymin,
        linewidth=0.5, edgecolor="orange", facecolor="none"
    )
    axes[1].add_patch(rect)
axes[1].set_title(f"Selective Search Proposals (showing 50 of {len(proposals)})", fontsize=12)
axes[1].axis("off")
plt.tight_layout()
plt.show()

# IoU analysis
best_ious = []
for gt_box in ann["boxes"]:
    best_iou = max(compute_iou(gt_box["bbox"], p) for p in proposals)
    best_ious.append(best_iou)
print(f"Best IoU per GT box: {[f'{iou:.2f}' for iou in best_ious]}")
print(f"Mean best IoU: {np.mean(best_ious):.2f}")

## 4. Feature Extraction per Proposal

For each proposal region, we extract visual features that our labeling functions will use to make decisions.

In [ ]:
# @title Feature Extraction
def extract_features(img, proposal):
    xmin, ymin, xmax, ymax = proposal
    xmin, ymin = max(0, xmin), max(0, ymin)
    xmax, ymax = min(img.shape[1], xmax), min(img.shape[0], ymax)
    if xmax <= xmin or ymax <= ymin:
        return {"area": 0, "aspect_ratio": 0, "mean_r": 0, "mean_g": 0, "mean_b": 0,
                "std_r": 0, "std_g": 0, "std_b": 0, "edge_density": 0, "center_x": 0, "center_y": 0}
    crop = img[ymin:ymax, xmin:xmax]
    area = crop.shape[0] * crop.shape[1]
    aspect_ratio = crop.shape[1] / max(crop.shape[0], 1)
    mean_rgb = crop.mean(axis=(0, 1))
    std_rgb = crop.std(axis=(0, 1))
    gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
    edges = cv2.Canny(gray, 50, 150)
    edge_density = edges.sum() / max(area, 1)
    img_h, img_w = img.shape[:2]
    return {
        "area": area,
        "aspect_ratio": aspect_ratio,
        "mean_r": mean_rgb[2],
        "mean_g": mean_rgb[1],
        "mean_b": mean_rgb[0],
        "std_r": std_rgb[2],
        "std_g": std_rgb[1],
        "std_b": std_rgb[0],
        "edge_density": edge_density,
        "center_x": (xmin + xmax) / 2 / img_w,
        "center_y": (ymin + ymax) / 2 / img_h,
    }

In [ ]:
# Build proposal dataset for a subset of images
N_IMAGES = 30  # Use subset for tutorial speed
MAX_PROPOSALS = 300

all_rows = []
all_filenames = list(all_annotations.keys())[:N_IMAGES]

for fname in tqdm(all_filenames, desc="Extracting features"):
    img_path = os.path.join(IMAGES_DIR, fname)
    img = cv2.imread(img_path)
    if img is None:
        continue
    proposals = generate_proposals(img, MAX_PROPOSALS)
    for prop in proposals:
        features = extract_features(img, prop)
        # Best GT box for this proposal
        best_iou = 0
        best_class = -1
        ann = all_annotations[fname]
        for gt_box in ann["boxes"]:
            iou = compute_iou(gt_box["bbox"], prop)
            if iou > best_iou:
                best_iou = iou
                best_class = gt_box["class_id"]
        row = {
            "filename": fname,
            "proposal": prop,
            **features,
            "best_iou": best_iou,
            "best_class": best_class if best_iou > 0.5 else -1,
        }
        all_rows.append(row)

df = pd.DataFrame(all_rows)
print(f"Total proposals: {len(df)}")
print(f"Proposals matching a GT box (IoU > 0.5): {(df['best_iou'] > 0.5).sum()}")
print(f"Positive proposals by class:")
pos_df = df[df["best_iou"] > 0.5]
if len(pos_df) > 0:
    print(pos_df["best_class"].map(CLASS_NAMES).value_counts())

# Feature distributions
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
features_to_plot = ["area", "aspect_ratio", "mean_r", "mean_g", "mean_b", "edge_density"]
for ax, feat in zip(axes.flat, features_to_plot):
    ax.hist(df[feat].values, bins=50, alpha=0.7)
    ax.set_title(feat)
    ax.set_yscale("log")
plt.tight_layout()
plt.show()

In [ ]:
# @title Compute LF thresholds from data statistics (no magic numbers)
# Compute LF thresholds from data statistics (no magic numbers)
pos_samples = df[df["best_iou"] > 0.5].copy()

if len(pos_samples) > 0:
    print("Per-class statistics for threshold computation:")
    class_stats = {}
    for cls_id, cls_name in CLASS_NAMES.items():
        cls_df = pos_samples[pos_samples["best_class"] == cls_id]
        if len(cls_df) > 0:
            means = cls_df[["mean_r", "mean_g", "mean_b"]].mean()
            total = means.sum() + 1e-10
            class_stats[cls_name] = {
                "n": len(cls_df),
                "brightness": means.sum(),
                "green_ratio": means["mean_g"] / total,
                "area_median": cls_df["area"].median(),
                "area_p25": cls_df["area"].quantile(0.25),
                "area_p75": cls_df["area"].quantile(0.75),
                "std_median": cls_df[["std_r", "std_g", "std_b"]].mean(axis=1).median(),
                "aspect_median": cls_df["aspect_ratio"].median(),
                "aspect_p25": cls_df["aspect_ratio"].quantile(0.25),
                "aspect_p75": cls_df["aspect_ratio"].quantile(0.75),
                "center_y_median": cls_df["center_y"].median(),
            }
            print(f"  {cls_name} (n={class_stats[cls_name]['n']}):")
            print(f"    Brightness={class_stats[cls_name]['brightness']:.0f}, Green ratio={class_stats[cls_name]['green_ratio']:.3f}")
            print(f"    Area: median={class_stats[cls_name]['area_median']:.0f}, p25={class_stats[cls_name]['area_p25']:.0f}, p75={class_stats[cls_name]['area_p75']:.0f}")
            print(f"    Aspect: median={class_stats[cls_name]['aspect_median']:.2f}, p25={class_stats[cls_name]['aspect_p25']:.2f}, p75={class_stats[cls_name]['aspect_p75']:.2f}")
            print(f"    Center Y: median={class_stats[cls_name]['center_y_median']:.3f}")

    # --- Compute data-driven thresholds ---
    aero = class_stats.get("aeroplane")
    boat = class_stats.get("boat")
    bottle = class_stats.get("bottle")
    plant = class_stats.get("pottedplant")

    # Aspect ratio: bottle is narrow, boat is wide
    if bottle and aero:
        NARROW_AR = (bottle["aspect_p75"] + aero["aspect_p25"]) / 2
    else:
        NARROW_AR = 0.6
    if boat and aero:
        WIDE_AR = (boat["aspect_p25"] + aero["aspect_p75"]) / 2
    else:
        WIDE_AR = 2.0

    # Area: bottle small, large for aero/boat
    if bottle and aero:
        SMALL_AREA = int((bottle["area_p75"] + aero["area_p25"]) / 2)
    else:
        SMALL_AREA = 3000

    # Green ratio for pottedplant
    if plant and aero:
        GREEN_RATIO_THRESH = (plant["green_ratio"] + aero["green_ratio"]) / 2
    else:
        GREEN_RATIO_THRESH = 0.36

    # Brightness for white objects
    if aero:
        BRIGHT_THRESH = aero["brightness"] * 0.85
    elif boat:
        BRIGHT_THRESH = boat["brightness"] * 0.85
    else:
        BRIGHT_THRESH = 550

    # Position: upper for aeroplane, lower for boat/pottedplant
    if aero:
        UPPER_Y_THRESH = aero["center_y_median"] * 1.3
    else:
        UPPER_Y_THRESH = 0.4
    if boat and plant:
        LOWER_Y_THRESH = (boat["center_y_median"] + plant["center_y_median"]) / 2 * 0.85
    else:
        LOWER_Y_THRESH = 0.55

    # Color std: white objects are uniform
    UNIFORM_STD = 35

    print(f"\nComputed thresholds:")
    print(f"  Narrow AR < {NARROW_AR:.2f} -> bottle")
    print(f"  Wide AR > {WIDE_AR:.2f} -> boat")
    print(f"  Small area < {SMALL_AREA} -> bottle")
    print(f"  Green ratio > {GREEN_RATIO_THRESH:.3f} -> pottedplant")
    print(f"  Brightness > {BRIGHT_THRESH:.0f} + upper (y<{UPPER_Y_THRESH:.2f}) -> aeroplane")
    print(f"  Brightness > {BRIGHT_THRESH:.0f} + lower (y>{LOWER_Y_THRESH:.2f}) + wide -> boat")
    print(f"  Color std < {UNIFORM_STD} + bright -> white object (aero/boat)")

    # Per-class RGB distribution plot
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for i, (channel, color) in enumerate([("mean_r", "red"), ("mean_g", "green"), ("mean_b", "blue")]):
        for cls_id, cls_name in CLASS_NAMES.items():
            cls_df = pos_samples[pos_samples["best_class"] == cls_id]
            if len(cls_df) > 0:
                axes[i].hist(cls_df[channel], bins=20, alpha=0.5, label=cls_name)
        axes[i].set_title(f"{color.title()} channel")
        axes[i].set_xlabel("Mean intensity")
        axes[i].legend()
    plt.suptitle("Per-class RGB distributions (thresholds computed from these)")
    plt.tight_layout()
    plt.show()
else:
    NARROW_AR = 0.6
    WIDE_AR = 2.0
    SMALL_AREA = 3000
    GREEN_RATIO_THRESH = 0.36
    BRIGHT_THRESH = 550
    UPPER_Y_THRESH = 0.4
    LOWER_Y_THRESH = 0.55
    UNIFORM_STD = 35
    print("No GT proposals available. Using default thresholds.")

## 5. Snorkel Labeling Functions

We write 6 labeling functions that classify each proposal based on visual heuristics.

**4 classes, 6 LFs:**

| LF | Heuristic | Logic |
|---|---|---|
| `lf_aspect_ratio` | Shape | Narrow AR < threshold → bottle, Wide AR > threshold → boat |
| `lf_green_dominant` | Color | G/(R+G+B) > threshold → pottedplant |
| `lf_bright_upper` | Color + Position | High brightness + upper half → aeroplane |
| `lf_white_boat` | Color + Position + Shape | High brightness + lower half + wide → boat |
| `lf_bright_uniform` | Color + Texture | High brightness + low color std → white object (aero/boat) |
| `lf_small_object` | Size | Small area + narrow-ish AR → bottle |

**ABSTAIN** means the LF has no opinion — Snorkel's LabelModel handles this.

In [ ]:
# Define label constants
AEROPLANE = 0
BOAT = 1
BOTTLE = 2
POTTEDPLANT = 3
ABSTAIN = -1

# Clean feature names for Snorkel (no spaces/dots)
df_lf = df.copy()
df_lf.columns = [c.replace(".", "_").replace(" ", "_") for c in df_lf.columns]

@labeling_function()
def lf_aspect_ratio(row):
    if row.aspect_ratio < NARROW_AR:
        return BOTTLE
    elif row.aspect_ratio > WIDE_AR:
        return BOAT
    return ABSTAIN

@labeling_function()
def lf_green_dominant(row):
    total = row.mean_r + row.mean_g + row.mean_b + 1e-10
    if total > 0 and (row.mean_g / total) > GREEN_RATIO_THRESH:
        return POTTEDPLANT
    return ABSTAIN

@labeling_function()
def lf_bright_upper(row):
    brightness = row.mean_r + row.mean_g + row.mean_b
    if brightness > BRIGHT_THRESH and row.center_y < UPPER_Y_THRESH:
        return AEROPLANE
    return ABSTAIN

@labeling_function()
def lf_white_boat(row):
    brightness = row.mean_r + row.mean_g + row.mean_b
    if brightness > BRIGHT_THRESH and row.center_y > LOWER_Y_THRESH and row.aspect_ratio > WIDE_AR:
        return BOAT
    return ABSTAIN

@labeling_function()
def lf_bright_uniform(row):
    brightness = row.mean_r + row.mean_g + row.mean_b
    mean_std = (row.std_r + row.std_g + row.std_b) / 3
    if brightness > BRIGHT_THRESH and mean_std < UNIFORM_STD:
        return AEROPLANE
    return ABSTAIN

@labeling_function()
def lf_small_object(row):
    if row.area < SMALL_AREA and row.aspect_ratio < 0.9:
        return BOTTLE
    return ABSTAIN

lfs = [lf_aspect_ratio, lf_green_dominant, lf_bright_upper, lf_white_boat, lf_bright_uniform, lf_small_object]

# Apply LFs
applier = PandasLFApplier(lfs)
L_train = applier.apply(df_lf)
print(f"Label matrix shape: {L_train.shape}")
print(f"Coverage per LF:")
lf_analysis = LFAnalysis(L_train)
for i, lf in enumerate(lfs):
    coverage = (L_train[:, i] != ABSTAIN).mean()
    print(f"  {lf.name}: {coverage:.1%} coverage")

In [ ]:
# Visualize LF outputs on proposals
def plot_lf_decisions(image_path, proposals_df, lf_idx, n_proposals=10):
    img = cv2.imread(image_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    fig, axes = plt.subplots(2, 5, figsize=(20, 8))
    sample_rows = proposals_df.sample(min(n_proposals, len(proposals_df)))
    for ax, (_, row) in zip(axes.flat, sample_rows.iterrows()):
        crop = img[row.proposal[1]:row.proposal[3], row.proposal[0]:row.proposal[2]]
        crop_rgb = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
        ax.imshow(crop_rgb)
        row_idx = df_lf.index.get_loc(row.name)
        lf_label = L_train[row_idx, lf_idx]
        label_name = CLASS_NAMES.get(lf_label, "ABSTAIN") if lf_label != -1 else "ABSTAIN"
        gt_name = CLASS_NAMES.get(row.best_class, "BG") if row.best_class != -1 else "BG"
        ax.set_title(f"LF: {label_name} | GT: {gt_name}", fontsize=9)
        ax.axis("off")
    plt.tight_layout()
    plt.show()

print("Showing decisions for lf_green_dominant (index 1):")
plot_lf_decisions(
    os.path.join(IMAGES_DIR, all_filenames[0]),
    df_lf[df_lf["filename"] == all_filenames[0]],
    lf_idx=1
)

## 6. Label Model

We train Snorkel's `LabelModel` to combine the labeling functions into a single probabilistic label per proposal.

In [ ]:
# @title Train LabelModel
label_model = LabelModel(cardinality=4, verbose=True)
label_model.fit(L_train, n_epochs=500, lr=0.01, l2=0.01, seed=123)

# Get probabilistic labels
probs = label_model.predict_proba(L_train)
preds = label_model.predict(L_train)

df_lf["snorkel_label"] = preds
df_lf["snorkel_confidence"] = probs.max(axis=1)
df_lf["snorkel_label_name"] = df_lf["snorkel_label"].map(CLASS_NAMES)

print("\nLabel distribution from LabelModel:")
print(df_lf["snorkel_label_name"].value_counts())

# Compare LabelModel vs majority vote vs individual LFs
majority_model = MajorityLabelVoter(cardinality=4)
majority_preds = majority_model.predict(L_train)

# Get accuracy where we have ground truth
gt_mask = df_lf["best_iou"] > 0.5
if gt_mask.sum() > 0:
    gt_labels = df_lf.loc[gt_mask, "best_class"].values
    lm_acc = (preds[gt_mask] == gt_labels).mean()
    mv_acc = (majority_preds[gt_mask] == gt_labels).mean()
    print(f"\nAccuracy (on proposals matching GT):")
    print(f"  LabelModel: {lm_acc:.1%}")
    print(f"  MajorityVote: {mv_acc:.1%}")

    # Individual LF accuracy
    print(f"  Individual LF accuracy:")
    for i, lf in enumerate(lfs):
        lf_mask = L_train[gt_mask, i] != ABSTAIN
        if lf_mask.sum() > 0:
            lf_acc = (L_train[gt_mask, i][lf_mask] == gt_labels[lf_mask]).mean()
            lf_cov = lf_mask.mean()
            print(f"    {lf.name}: acc={lf_acc:.1%}, cov={lf_cov:.1%}")

    # Confusion matrix: LabelModel predictions vs ground truth
    fig, ax = plt.subplots(figsize=(8, 6))
    cm = confusion_matrix(gt_labels, preds[gt_mask], labels=[0, 1, 2, 3])
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Aeroplane", "Boat", "Bottle", "Pottedplant"])
    disp.plot(ax=ax, cmap="Blues", values_format="d")
    ax.set_title("LabelModel Predictions vs Ground Truth")
    plt.tight_layout()
    plt.show()

In [ ]:
# LFAnalysis details
# Call without Y (ground truth) since not all proposals have GT labels
lf_summary = lf_analysis.lf_summary()
print("\nLabeling Function Summary (coverage, conflicts, polarity):")
print(lf_summary)

# Plot LF statistics
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
lf_summary["Coverage"].plot(kind="bar", ax=axes[0])
axes[0].set_title("LF Coverage")
axes[0].set_ylabel("Coverage")
axes[0].tick_params(axis="x", rotation=45)

lf_summary["Conflicts"].plot(kind="bar", ax=axes[1])
axes[1].set_title("LF Conflicts")
axes[1].set_ylabel("Conflicts")
axes[1].tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

# Pairwise LF agreement heatmap
n_lfs = L_train.shape[1]
agreement = np.zeros((n_lfs, n_lfs))
for i in range(n_lfs):
    for j in range(n_lfs):
        both_voted = (L_train[:, i] != ABSTAIN) & (L_train[:, j] != ABSTAIN)
        if both_voted.sum() > 0:
            agreement[i, j] = (L_train[both_voted, i] == L_train[both_voted, j]).mean()
        else:
            agreement[i, j] = 0

fig, ax = plt.subplots(figsize=(8, 6))
lf_names = [lf.name.replace("lf_", "") for lf in lfs]
sns.heatmap(agreement, annot=True, fmt=".2f", xticklabels=lf_names, yticklabels=lf_names,
            cmap="YlOrRd", vmin=0, vmax=1, ax=ax)
ax.set_title("Pairwise LF Agreement (when both vote)")
plt.tight_layout()
plt.show()

# Polarity (which classes each LF outputs)
print("\nLF Polarities (output classes):")
lf_polarities = lf_analysis.lf_polarities()
for i, lf in enumerate(lfs):
    print(f"  {lf.name}: {lf_polarities[i]}")

In [ ]:
# Confidence distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(df_lf["snorkel_confidence"], bins=50)
axes[0].set_title("LabelModel Confidence Distribution")
axes[0].set_xlabel("Confidence")
axes[0].set_ylabel("Count")

# Per-class confidence
for cls_id, cls_name in CLASS_NAMES.items():
    cls_conf = df_lf[df_lf["snorkel_label"] == cls_id]["snorkel_confidence"]
    axes[1].hist(cls_conf, bins=30, alpha=0.5, label=cls_name)
axes[1].set_title("Confidence by Class")
axes[1].set_xlabel("Confidence")
axes[1].set_ylabel("Count")
axes[1].legend()
plt.tight_layout()
plt.show()

## 7. Pseudo-Label Assignment

We assign weak labels to proposals and apply Non-Maximum Suppression (NMS) to remove redundant boxes.

In [ ]:
# @title Assign Pseudo-Labels
CONFIDENCE_THRESHOLD = 0.7

# Filter high-confidence positive proposals
pos_mask = df_lf["snorkel_confidence"] >= CONFIDENCE_THRESHOLD
pos_df = df_lf[pos_mask].copy()

print(f"High-confidence proposals: {len(pos_df)} out of {len(df_lf)}")
print(f"\nClass distribution:")
print(pos_df["snorkel_label_name"].value_counts())

In [ ]:
# Non-Maximum Suppression
def nms(proposals, scores, iou_threshold=0.5):
    if len(proposals) == 0:
        return []
    boxes_xywh = [[b[0], b[1], b[2] - b[0], b[3] - b[1]] for b in proposals]
    scores_arr = np.array(scores)
    indices = cv2.dnn.NMSBoxes(
        boxes_xywh, scores_arr.tolist(), score_threshold=0.0, nms_threshold=iou_threshold
    )
    if indices is None or len(indices) == 0:
        return []
    if hasattr(indices, "flatten"):
        return indices.flatten().tolist()
    if len(indices.shape) > 1:
        return indices[:, 0].tolist()
    return indices.tolist()

def apply_nms_per_image(df, score_col="snorkel_confidence", iou_threshold=0.5):
    kept_rows = []
    for fname in df["filename"].unique():
        img_df = df[df["filename"] == fname]
        proposals_list = img_df["proposal"].tolist()
        scores_list = img_df[score_col].tolist()
        keep = nms(proposals_list, scores_list, iou_threshold)
        kept_idxs = img_df.index[keep]
        kept_rows.extend(img_df.loc[kept_idxs].to_dict("records"))
    return pd.DataFrame(kept_rows)

pos_df_nms = apply_nms_per_image(pos_df, iou_threshold=0.5)
print(f"After NMS: {len(pos_df_nms)} proposals (from {len(pos_df)})")

In [ ]:
# Visualize pseudo-labels vs GT
def visualize_pseudo_labels(fname, ax, pos_df, title_prefix=""):
    img_path = os.path.join(IMAGES_DIR, fname)
    img = cv2.imread(img_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    ax.imshow(img_rgb)

    # GT boxes
    ann = all_annotations[fname]
    for box in ann["boxes"]:
        xmin, ymin, xmax, ymax = box["bbox"]
        rect = patches.Rectangle(
            (xmin, ymin), xmax - xmin, ymax - ymin,
            linewidth=1.5, edgecolor="lime", facecolor="none", linestyle="--"
        )
        ax.add_patch(rect)

    # Pseudo-label boxes
    img_df = pos_df[pos_df["filename"] == fname]
    colors = {"aeroplane": "cyan", "boat": "yellow", "bottle": "magenta", "pottedplant": "lime"}
    for _, row in img_df.iterrows():
        prop = row["proposal"]
        rect = patches.Rectangle(
            (prop[0], prop[1]), prop[2] - prop[0], prop[3] - prop[1],
            linewidth=2, edgecolor=colors.get(row["snorkel_label_name"], "red"),
            facecolor="none"
        )
        ax.add_patch(rect)
        ax.text(prop[0], prop[1] - 8, f"{row['snorkel_label_name']}:{row['snorkel_confidence']:.2f}",
                color=colors.get(row["snorkel_label_name"], "red"), fontsize=8)

    ax.set_title(f"{title_prefix}{fname} (GT=--, Pseudo=-)")
    ax.axis("off")

sample_files_vis = [f for f in all_filenames if f in pos_df_nms["filename"].values][:4]
if len(sample_files_vis) < 4:
    sample_files_vis = list(pos_df_nms["filename"].unique())[:4]

if len(sample_files_vis) > 0:
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    for ax, fname in zip(axes.flat, sample_files_vis):
        visualize_pseudo_labels(fname, ax, pos_df_nms)
    plt.tight_layout()
    plt.show()

# Quality assessment
pos_gt = pos_df_nms[pos_df_nms["best_iou"] > 0.5]
bg_gt = pos_df_nms[pos_df_nms["best_iou"] <= 0.5]
print(f"\nQuality of pseudo-labels:")
print(f"  Correct (IoU>0.5 with GT): {len(pos_gt)} ({len(pos_gt)/max(len(pos_df_nms),1):.1%})")
print(f"  Background (no match): {len(bg_gt)} ({len(bg_gt)/max(len(pos_df_nms),1):.1%})")
if len(pos_gt) > 0:
    correct_class = (pos_gt["snorkel_label"] == pos_gt["best_class"]).mean()
    print(f"  Class accuracy (on matched): {correct_class:.1%}")

## 8. COCO Format Conversion

We convert the pseudo-labels to COCO JSON format for DETR training.

In [ ]:
# @title Build COCO Dataset
def build_coco_json(pos_df, filenames, output_path="pseudo_annotations.json"):
    coco = {
        "images": [],
        "annotations": [],
        "categories": [
            {"id": 0, "name": "aeroplane"},
            {"id": 1, "name": "boat"},
            {"id": 2, "name": "bottle"},
            {"id": 3, "name": "pottedplant"},
        ],
    }
    ann_id = 0
    for img_id, fname in enumerate(filenames):
        img_path = os.path.join(IMAGES_DIR, fname)
        img = cv2.imread(img_path)
        if img is None:
            continue
        h, w = img.shape[:2]
        coco["images"].append({
            "id": img_id, "file_name": fname, "width": w, "height": h
        })
        img_df = pos_df[pos_df["filename"] == fname]
        for _, row in img_df.iterrows():
            prop = row["proposal"]
            xmin, ymin, xmax, ymax = prop
            bw = xmax - xmin
            bh = ymax - ymin
            if bw <= 0 or bh <= 0:
                continue
            coco["annotations"].append({
                "id": ann_id,
                "image_id": img_id,
                "category_id": int(row["snorkel_label"]),
                "bbox": [int(xmin), int(ymin), int(bw), int(bh)],
                "area": int(bw * bh),
                "iscrowd": 0,
                "score": float(row["snorkel_confidence"]),
            })
            ann_id += 1

    os.makedirs(os.path.dirname(output_path) if os.path.dirname(output_path) else ".", exist_ok=True)
    with open(output_path, "w") as f:
        json.dump(coco, f)
    print(f"Saved {len(coco['annotations'])} annotations for {len(coco['images'])} images to {output_path}")
    return coco

coco_data = build_coco_json(pos_df_nms, all_filenames, "weak_annotations/coco.json")

# Also build ground truth COCO for evaluation
def build_gt_coco(filenames, output_path="gt_annotations.json"):
    coco = {
        "images": [],
        "annotations": [],
        "categories": [
            {"id": 0, "name": "aeroplane"},
            {"id": 1, "name": "boat"},
            {"id": 2, "name": "bottle"},
            {"id": 3, "name": "pottedplant"},
        ],
    }
    ann_id = 0
    for img_id, fname in enumerate(filenames):
        img_path = os.path.join(IMAGES_DIR, fname)
        img = cv2.imread(img_path)
        if img is None:
            continue
        h, w = img.shape[:2]
        coco["images"].append({"id": img_id, "file_name": fname, "width": w, "height": h})
        ann = all_annotations[fname]
        for box in ann["boxes"]:
            xmin, ymin, xmax, ymax = box["bbox"]
            bw, bh = xmax - xmin, ymax - ymin
            coco["annotations"].append({
                "id": ann_id, "image_id": img_id, "category_id": box["class_id"],
                "bbox": [xmin, ymin, bw, bh], "area": bw * bh, "iscrowd": 0,
            })
            ann_id += 1
    with open(output_path, "w") as f:
        json.dump(coco, f)
    print(f"Saved {len(coco['annotations'])} GT annotations for {len(coco['images'])} images to {output_path}")
    return coco

_ = build_gt_coco(all_filenames, "weak_annotations/gt.json")

In [ ]:
# Create PyTorch Dataset for DETR
class VOCDataset(Dataset):
    def __init__(self, coco_json_path, img_dir, image_processor):
        with open(coco_json_path) as f:
            self.coco = json.load(f)
        self.img_dir = img_dir
        self.image_processor = image_processor
        self.image_id_to_data = {img["id"]: img for img in self.coco["images"]}
        self.annotations_by_image = {}
        for ann in self.coco["annotations"]:
            img_id = ann["image_id"]
            if img_id not in self.annotations_by_image:
                self.annotations_by_image[img_id] = []
            self.annotations_by_image[img_id].append(ann)

    def __len__(self):
        return len(self.coco["images"])

    def __getitem__(self, idx):
        img_info = self.coco["images"][idx]
        img_path = os.path.join(self.img_dir, img_info["file_name"])
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        annotations = self.annotations_by_image.get(img_info["id"], [])

        # Build COCO-format annotations list
        coco_annotations = []
        for ann in annotations:
            coco_annotations.append({
                "bbox": ann["bbox"],
                "category_id": ann["category_id"],
                "area": ann["bbox"][2] * ann["bbox"][3],
                "iscrowd": 0,
            })

        encoding = self.image_processor(
            images=image,
            annotations={"image_id": img_info["id"], "annotations": coco_annotations},
            return_tensors="pt",
        )
        # Remove batch dimension added by processor
        for k in encoding:
            if isinstance(encoding[k], torch.Tensor) and encoding[k].dim() > 0:
                encoding[k] = encoding[k].squeeze(0)
        encoding["orig_size"] = torch.tensor([image.shape[0], image.shape[1]])
        encoding["image_id"] = img_info["id"]
        return encoding

processor = DetrImageProcessor.from_pretrained("facebook/detr-resnet-50")

# Split into train/val
random.seed(42)
all_ids = sorted(coco_data["images"], key=lambda x: x["id"])
random.shuffle(all_ids)
split_idx = int(len(all_ids) * 0.8)
train_ids = all_ids[:split_idx]
val_ids = all_ids[split_idx:]

def subset_coco(coco_data, image_ids_subset):
    subset = {
        "images": image_ids_subset,
        "categories": coco_data["categories"],
        "annotations": [a for a in coco_data["annotations"] if a["image_id"] in {i["id"] for i in image_ids_subset}],
    }
    return subset

def save_subset_coco(coco_subset, path):
    with open(path, "w") as f:
        json.dump(coco_subset, f)

os.makedirs("weak_annotations", exist_ok=True)
train_coco = subset_coco(coco_data, train_ids)
val_coco = subset_coco(coco_data, val_ids)
save_subset_coco(train_coco, "weak_annotations/train_coco.json")
save_subset_coco(val_coco, "weak_annotations/val_coco.json")

train_dataset = VOCDataset("weak_annotations/train_coco.json", IMAGES_DIR, processor)
val_dataset = VOCDataset("weak_annotations/val_coco.json", IMAGES_DIR, processor)

print(f"Train dataset: {len(train_dataset)} images, "
      f"{sum(len(v) for v in train_dataset.annotations_by_image.values())} annotations")
print(f"Val dataset: {len(val_dataset)} images, "
      f"{sum(len(v) for v in val_dataset.annotations_by_image.values())} annotations")

## 9. DETR Fine-Tuning

We fine-tune a pretrained DETR model using our weakly-labeled data.

In [ ]:
# @title Initialize DETR Model
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")

model = DetrForObjectDetection.from_pretrained(
    "facebook/detr-resnet-50",
    num_labels=4,
    ignore_mismatched_sizes=True,
)
model.to(device)
print(f"Model loaded: {sum(p.numel() for p in model.parameters()):,} parameters")

In [ ]:
# @title Training Arguments
training_args = TrainingArguments(
    output_dir="detr_voc_weak",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=10,
    eval_strategy="steps",
    eval_steps=20,
    logging_steps=10,
    save_steps=50,
    save_total_limit=2,
    learning_rate=5e-5,
    weight_decay=1e-4,
    remove_unused_columns=False,
    report_to="none",
    fp16=torch.cuda.is_available(),
    dataloader_num_workers=0,
    push_to_hub=False,
)

def collate_fn(batch, include_extra=False):
    pixel_values_list = [item["pixel_values"] for item in batch]
    pixel_mask_list = [item["pixel_mask"] for item in batch]
    max_h = max(pv.shape[1] for pv in pixel_values_list)
    max_w = max(pv.shape[2] for pv in pixel_values_list)
    batch_size = len(batch)
    pixel_values = torch.zeros(batch_size, 3, max_h, max_w, dtype=pixel_values_list[0].dtype)
    pixel_mask = torch.zeros(batch_size, max_h, max_w, dtype=pixel_mask_list[0].dtype)
    for i, (pv, pm) in enumerate(zip(pixel_values_list, pixel_mask_list)):
        h, w = pv.shape[1], pv.shape[2]
        pixel_values[i, :, :h, :w] = pv
        pixel_mask[i, :h, :w] = pm
    labels = [item["labels"][0] for item in batch]
    result = {"pixel_values": pixel_values, "pixel_mask": pixel_mask, "labels": labels}
    if include_extra:
        orig_sizes = torch.stack([item["orig_size"] for item in batch])
        image_ids = [item["image_id"] for item in batch]
        result["orig_sizes"] = orig_sizes
        result["image_ids"] = image_ids
    return result

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=collate_fn,
    tokenizer=processor,
)

print("Trainer initialized. Starting training...")
print("(Training may take 5-15 minutes depending on GPU)")
results = trainer.train()

print("\nTraining complete!")
print(f"Final loss: {results.training_loss:.4f}")

# Plot training loss curve
log_history = trainer.state.log_history
train_steps = [entry["step"] for entry in log_history if "loss" in entry and "eval_loss" not in entry]
train_losses = [entry["loss"] for entry in log_history if "loss" in entry and "eval_loss" not in entry]
eval_steps = [entry["step"] for entry in log_history if "eval_loss" in entry]
eval_losses = [entry["eval_loss"] for entry in log_history if "eval_loss" in entry]

fig, ax = plt.subplots(figsize=(10, 5))
if train_steps:
    ax.plot(train_steps, train_losses, "b-o", markersize=3, label="Training loss")
if eval_steps:
    ax.plot(eval_steps, eval_losses, "r-s", markersize=4, label="Validation loss")
ax.set_xlabel("Training step")
ax.set_ylabel("Loss")
ax.set_title("DETR Training Progress")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

if eval_losses:
    print(f"Best validation loss: {min(eval_losses):.4f} at step {eval_steps[eval_losses.index(min(eval_losses))]}")
print(f"Total training steps: {len(train_steps)}")

In [ ]:
# Save the fine-tuned model
model.save_pretrained("detr_voc_weak/final")
processor.save_pretrained("detr_voc_weak/final")
print("Model saved to detr_voc_weak/final/")

## 10. Evaluation

We evaluate our weakly-supervised model using mAP and visualize predictions.

In [ ]:
# @title Evaluation with pycocotools
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

# Load GT annotations
coco_gt = COCO("weak_annotations/gt.json")

# We'll evaluate on our validation set
val_fnames = {img["file_name"] for img in val_coco["images"]}
val_img_ids = [img["id"] for img in val_coco["images"]]

In [ ]:
# @title Run inference on validation set
model.eval()
results_list = []

with torch.no_grad():
    for batch in DataLoader(val_dataset, batch_size=4, collate_fn=partial(collate_fn, include_extra=True)):
        pixel_values = batch["pixel_values"].to(device)
        pixel_mask = batch["pixel_mask"].to(device)
        orig_sizes = batch["orig_sizes"].to(device)
        image_ids = batch["image_ids"]
        outputs = model(pixel_values=pixel_values, pixel_mask=pixel_mask)

        processed = processor.post_process_object_detection(
            outputs, threshold=0.3, target_sizes=orig_sizes
        )

        for i, result in enumerate(processed):
            for score, label, box in zip(result["scores"], result["labels"], result["boxes"]):
                results_list.append({
                    "image_id": image_ids[i],
                    "category_id": int(label),
                    "bbox": box.tolist(),
                    "score": float(score),
                })
                # Limit results to avoid memory issues
                if len(results_list) >= 1000:
                    break
            if len(results_list) >= 1000:
                break
        if len(results_list) >= 1000:
            break

coco_dt = coco_gt.loadRes(results_list)
coco_eval = COCOeval(coco_gt, coco_dt, iouType="bbox")
coco_eval.params.imgIds = val_img_ids
coco_eval.evaluate()
coco_eval.accumulate()
coco_eval.summarize()

print(f"\n{'='*50}")
print(f"Final Results:")
print(f"  mAP@0.5: {coco_eval.stats[1]:.3f}")
print(f"  mAP@0.5:0.95: {coco_eval.stats[0]:.3f}")

# Per-class metrics
print(f"\nPer-class AP@0.5:")
for cat in coco_gt.dataset["categories"]:
    cat_id = cat["id"]
    coco_eval.params.catIds = [cat_id]
    coco_eval.params.imgIds = val_img_ids
    coco_eval.evaluate()
    coco_eval.accumulate()
    ap50 = coco_eval.stats[1]
    print(f"  {cat['name']}: {ap50:.3f}")

# Precision-Recall curves per class
print("\nPlotting Precision-Recall curves...")
fig, ax = plt.subplots(figsize=(10, 8))
colors_pr = ["cyan", "yellow", "magenta", "lime"]
for cat in coco_gt.dataset["categories"]:
    cat_id = cat["id"]
    gt_arr = np.zeros(len(val_dataset))
    pred_scores = np.zeros(len(val_dataset))
    for i, img in enumerate(val_dataset.coco["images"]):
        img_anns = [a for a in val_dataset.coco["annotations"] if a["image_id"] == img["id"]]
        gt_arr[i] = 1 if any(a["category_id"] == cat_id for a in img_anns) else 0
        matches = [r for r in results_list if r["image_id"] == img["id"] and r["category_id"] == cat_id]
        pred_scores[i] = max([r["score"] for r in matches]) if matches else 0
    if gt_arr.sum() > 0 and len(np.unique(gt_arr)) > 1:
        precision, recall, _ = precision_recall_curve(gt_arr, pred_scores)
        ap = average_precision_score(gt_arr, pred_scores)
        ax.plot(recall, precision, color=colors_pr[cat_id], lw=2,
                label=f"{cat['name']} (AP={ap:.2f})")
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title("Precision-Recall Curves per Class")
ax.legend(loc="lower left")
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()

In [ ]:
# Visualize predictions
model.eval()
sample_val_files = [f for f in all_filenames if f in val_fnames][:10]

n_cols = 5
n_rows = (len(sample_val_files) + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, 4 * n_rows))
colors = {0: "cyan", 1: "yellow", 2: "magenta", 3: "lime"}

for ax, fname in zip(axes.flat, sample_val_files):
    img_path = os.path.join(IMAGES_DIR, fname)
    img = cv2.imread(img_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    ax.imshow(img_rgb)

    # GT boxes
    ann = all_annotations[fname]
    for box in ann["boxes"]:
        xmin, ymin, xmax, ymax = box["bbox"]
        rect = patches.Rectangle(
            (xmin, ymin), xmax - xmin, ymax - ymin,
            linewidth=1.5, edgecolor="lime", facecolor="none", linestyle="--"
        )
        ax.add_patch(rect)

    # Run inference
    encoding = processor(images=img_rgb, return_tensors="pt")
    encoding = {k: v.to(device) for k, v in encoding.items()}
    with torch.no_grad():
        outputs = model(**encoding)

    target_sizes = torch.tensor([[img_rgb.shape[0], img_rgb.shape[1]]]).to(device)
    results = processor.post_process_object_detection(
        outputs, threshold=0.3, target_sizes=target_sizes
    )[0]

    for score, label, box in zip(results["scores"], results["labels"], results["boxes"]):
        box = box.cpu().tolist()
        xmin, ymin, xmax, ymax = box
        rect = patches.Rectangle(
            (xmin, ymin), xmax - xmin, ymax - ymin,
            linewidth=2, edgecolor=colors.get(int(label), "red"), facecolor="none"
        )
        ax.add_patch(rect)
        ax.text(xmin, ymin - 8, f"{CLASS_NAMES[int(label)]}:{score:.2f}",
                color=colors.get(int(label), "red"), fontsize=9)

    ax.set_title(fname)
    ax.axis("off")

plt.suptitle("Predictions (GT=--lime, Predicted=Cyan/Yellow/Magenta)", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Error Analysis: False Positives and False Negatives
print("\n--- Error Analysis ---")
print(f"\nDetection summary:")
print(f"  Total GT objects in val set: {sum(len(v) for v in val_dataset.annotations_by_image.values())}")
print(f"  Total predictions: {len(results_list)}")
print(f"  Predictions per class:")
for cat_id, cat_name in CLASS_NAMES.items():
    n_pred = sum(1 for r in results_list if r["category_id"] == cat_id)
    print(f"    {cat_name}: {n_pred}")

# Show images with most false positives
img_pred_counts = {}
for r in results_list:
    img_pred_counts[r["image_id"]] = img_pred_counts.get(r["image_id"], 0) + 1
if img_pred_counts:
    worst_img_id = max(img_pred_counts, key=img_pred_counts.get)
    worst_img = [i for i in val_dataset.coco["images"] if i["id"] == worst_img_id][0]
    print(f"\nImage with most predictions: {worst_img['file_name']} ({img_pred_counts[worst_img_id]} preds)")

# Confidence histogram of predictions
if results_list:
    fig, ax = plt.subplots(figsize=(10, 4))
    scores = [r["score"] for r in results_list]
    ax.hist(scores, bins=30, alpha=0.7, color="steelblue", edgecolor="white")
    ax.set_xlabel("Detection confidence")
    ax.set_ylabel("Count")
    ax.set_title(f"Prediction Confidence Distribution ({len(results_list)} total detections)")
    ax.axvline(x=0.5, color="red", linestyle="--", alpha=0.5, label="Confidence threshold=0.3")
    ax.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
# Final summary
print("\n" + "="*60)
print("SNORKEL + DETR FOR PASCAL VOC OBJECT DETECTION: SUMMARY")
print("="*60)
print(f"\nDataset: Pascal VOC 2012 ({len(all_annotations)} images, 4 classes)")
print(f"Region proposals: Selective Search (up to {MAX_PROPOSALS} per image)")
print(f"Labeling functions: {len(lfs)} heuristic LFs")
print(f"  - lf_aspect_ratio, lf_green_dominant, lf_bright_upper")
print(f"  - lf_white_boat, lf_bright_uniform, lf_small_object")
print(f"LabelModel: Trained on {len(df_lf)} proposals")
print(f"  - High-confidence pseudo-labels: {len(pos_df_nms)} (thresh={CONFIDENCE_THRESHOLD})")
print(f"  - Quality: {(len(pos_gt)/max(len(pos_df_nms),1)):.1%} match GT objects")
print(f"DETR Model: facebook/detr-resnet-50 fine-tuned on pseudo-labels")
print(f"  - Training: 10 epochs on weak labels")
print(f"  - Eval: mAP@0.5 = {coco_eval.stats[1]:.3f}")

print(f"\n{'='*60}")
print("TUTORIAL COMPLETE!")
print("="*60)

## Conclusion

This tutorial demonstrated how to use **Snorkel** for programmatic data labeling in an object detection pipeline:

1. **Region proposals** via Selective Search generated candidate boxes
2. **Labeling functions** encoded domain knowledge about Pascal VOC object appearance
3. **LabelModel** combined weak signals into probabilistic labels
4. **Pseudo-labels** were filtered and used to train a **DETR** object detector
5. **Evaluation** showed the model learned meaningful detection from weak supervision

### Next Steps
- Add more labeling functions targeting specific failure modes
- Use active learning to prioritize proposals the LabelModel is uncertain about
- Scale to more Pascal VOC 2012 images (~11k available)
- Try different backbones (YOLO, Faster R-CNN) with the same weak labels